# 03 - Exploratory Data Analysis (EDA)

## Doel
Inzichten halen uit de data:
- Goudprijstrends en aankooptiming
- Samenstelling van de portefeuille
- Correlaties tussen variabelen
- Beantwoord de vraag: heb ik op goede momenten gekocht?

In [ ]:
# Cel 1: Inladen van bibliotheken en instellen van paden

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import (
    load_portfolio, calculate_pure_gold_weight,
    filter_bars, filter_jewelry,
    calculate_portfolio_value, goal_progress, next_purchase_date,
    format_eur, format_gram, KARAT_PURITY
)
from src.data_loader import load_raw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_palette('husl')

PROC_DIR = Path('../data/processed')
print('Setup complete.')

In [ ]:
# Cel 2: Beantwoord de vraag: heb ik op goede momenten gekocht?

# Laad schone data
gold = pd.read_csv(PROC_DIR / 'gold_with_macro.csv', index_col=0, parse_dates=True)
portfolio = pd.read_csv(PROC_DIR / 'portfolio_cleaned.csv')
portfolio['datum_aankoop'] = pd.to_datetime(portfolio['datum_aankoop'])
bars = pd.read_csv(PROC_DIR / 'bars_with_spot.csv')
bars['datum_aankoop'] = pd.to_datetime(bars['datum_aankoop'])

print(f'Gold data: {len(gold)} rijen')
print(f'Portfolio: {len(portfolio)} items')
print(f'Goudstaven: {len(bars)} items')

## 1. Goudprijstrend (2020 - nu)

In [ ]:
# Cel 3: Goudprijstrend (2020 - nu)

fig, ax1 = plt.subplots(figsize=(16, 7))

# Goudprijs in USD
color1 = 'gold'
ax1.plot(gold.index, gold['gold_usd_oz'], color=color1, linewidth=1.5, label='Goud (USD/oz)')
ax1.fill_between(gold.index, gold['gold_usd_oz'], alpha=0.15, color=color1)
ax1.set_ylabel('Goudprijs (USD/oz)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)

# EUR/USD op secundaire as
if 'eur_usd' in gold.columns:
    ax2 = ax1.twinx()
    color2 = 'steelblue'
    ax2.plot(gold.index, gold['eur_usd'], color=color2, linewidth=1, alpha=0.6, label='EUR/USD')
    ax2.set_ylabel('EUR/USD', color=color2, fontsize=12)
    ax2.tick_params(axis='y', labelcolor=color2)

# Aankoop markers
for _, row in bars.iterrows():
    ax1.axvline(row['datum_aankoop'], color='red', linestyle='--', alpha=0.3, linewidth=0.8)
    ax1.scatter(row['datum_aankoop'], row.get('spot_prijs_aankoop', 0) * 31.1035 if 'spot_prijs_aankoop' in bars.columns else 0,
               color='red', s=80, zorder=5, edgecolors='black')

ax1.set_title('Goudprijs & EUR/USD Wisselkoers (2020 - nu)\nMet aankoopmomenten (rood)', fontsize=14)
ax1.set_xlabel('Datum')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels() if 'eur_usd' in gold.columns else ([], [])
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

## 2. Portefeuille Samenstelling

In [ ]:
# Cel 4: Portefeuille Samenstelling

portfolio_calc = calculate_pure_gold_weight(portfolio)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Karat verdeling
karat_counts = portfolio_calc['karaat'].value_counts().sort_index()
axes[0, 0].pie(karat_counts.values, labels=[f'{k}k' for k in karat_counts.index],
               autopct='%1.0f%%', colors=sns.color_palette('YlOrRd', len(karat_counts)))
axes[0, 0].set_title('Verdeling per Karaat')

# Type verdeling
type_counts = portfolio_calc['type'].value_counts()
axes[0, 1].bar(type_counts.index, type_counts.values, color=['gold', 'silver'])
axes[0, 1].set_title('Verdeling: Staven vs Sieraden')
axes[0, 1].set_ylabel('Aantal items')

# Gewicht per type
weight_by_type = portfolio_calc.groupby('type')['gram'].sum()
axes[1, 0].bar(weight_by_type.index, weight_by_type.values, color=['gold', 'silver'])
axes[1, 0].set_title('Totaal Gewicht per Type')
axes[1, 0].set_ylabel('Gram')

# Zuiver goud per karaat
pure_by_karat = portfolio_calc.groupby('karaat')['zuiver_goud_gram'].sum()
axes[1, 1].bar([f'{k}k' for k in pure_by_karat.index], pure_by_karat.values,
               color=sns.color_palette('YlOrRd', len(pure_by_karat)))
axes[1, 1].set_title('Zuiver Goud per Karaat')
axes[1, 1].set_ylabel('Zuiver goud (gram)')

plt.tight_layout()
plt.show()

In [ ]:
# Cel 5: Portefeuille Samenstelling

# Top 10 zwaarste items
print('--- Top 10 Zwaarste Items ---')
top10 = portfolio_calc.nlargest(10, 'gram')[['omschrijving', 'type', 'karaat', 'gram', 'zuiver_goud_gram']]
top10.to_string(index=False)

## 3. Aankooptiming Analyse

Heb ik op goede momenten goud gekocht?

In [ ]:
# Cel 6: Heb ik op goede momenten goud gekocht?

if 'prijs_per_gram' in bars.columns:
    fig, ax = plt.subplots(figsize=(14, 6))

    # Spot prijs lijn
    if 'gold_eur_gram' in gold.columns:
        ax.plot(gold.index, gold['gold_eur_gram'], color='gray', linewidth=1,
               alpha=0.7, label='Spot prijs (EUR/gram)')

    # Aankoop punten
    ax.scatter(bars['datum_aankoop'], bars['prijs_per_gram'],
              color='red', s=100, zorder=5, edgecolors='black', label='Aankoopprijs')

    # Gemiddelde aankoopprijs
    gem = bars['prijs_per_gram'].mean()
    ax.axhline(gem, color='red', linestyle='--', alpha=0.5, label=f'Gem. aankoopprijs: EUR {gem:.2f}')

    if 'gold_eur_gram' in gold.columns:
        gem_spot = gold['gold_eur_gram'].mean()
        ax.axhline(gem_spot, color='gray', linestyle='--', alpha=0.5, label=f'Gem. spot: EUR {gem_spot:.2f}')

    ax.set_title('Aankoopprijs vs Spot Prijs (per gram)', fontsize=14)
    ax.set_ylabel('EUR per gram')
    ax.set_xlabel('Datum')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('prijs_per_gram kolom niet gevonden - voer Notebook 02 eerst uit')

In [ ]:
# Cel 7: Heb ik op goede momenten goud gekocht?

# Statistieken: aankoop vs spot
if 'prijs_per_gram' in bars.columns:
    print('--- Aankoop vs Spot Statistieken ---')
    print(f"Gem. aankoopprijs/gram:  EUR {bars['prijs_per_gram'].mean():.2f}")
    print(f"Med. aankoopprijs/gram:  EUR {bars['prijs_per_gram'].median():.2f}")
    print(f"Std. aankoopprijs/gram:  EUR {bars['prijs_per_gram'].std():.2f}")
    print()
    print(f"Gem. spot prijs/gram:    EUR {bars['spot_prijs_aankoop'].mean():.2f}")
    print(f"Med. spot prijs/gram:    EUR {bars['spot_prijs_aankoop'].median():.2f}")
    print()
    print(f"Gem. premium (verschil): EUR {bars['verschil_vs_spot'].mean():.2f} per gram")
    print(f"Med. premium:            EUR {bars['verschil_vs_spot'].median():.2f} per gram")

## 4. Maandelijkse Goudprijs Analyse

In [ ]:
# Cel 8: Maandelijkse Goudprijs Analyse

# Maandelijkse gemiddelde goudprijs
monthly = gold['gold_eur_gram'].resample('ME').agg(['mean', 'std', 'min', 'max']) if 'gold_eur_gram' in gold.columns else gold['gold_usd_oz'].resample('ME').agg(['mean', 'std', 'min', 'max'])

col_name = 'gold_eur_gram' if 'gold_eur_gram' in gold.columns else 'gold_usd_oz'
unit = 'EUR/gram' if 'gold_eur_gram' in gold.columns else 'USD/oz'

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(monthly.index, monthly['min'], monthly['max'], alpha=0.2, color='gold')
ax.plot(monthly.index, monthly['mean'], color='gold', linewidth=2, label=f'Gemiddelde {unit}')
ax.plot(monthly.index, monthly['mean'] + monthly['std'], color='orange', linestyle='--', alpha=0.5, label='+1 std')
ax.plot(monthly.index, monthly['mean'] - monthly['std'], color='orange', linestyle='--', alpha=0.5, label='-1 std')

# Aankoop markers
for _, row in bars.iterrows():
    ax.axvline(row['datum_aankoop'], color='red', linestyle='--', alpha=0.2)

ax.set_title(f'Maandelijkse Goudprijs ({unit}) met Variatiewijdte', fontsize=14)
ax.set_ylabel(unit)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Correlatie Analyse

In [ ]:
# Cel 9: Correlatie Analyse

# Selecteer numerieke kolommen voor correlatie
num_cols = gold.select_dtypes(include=[np.number]).columns.tolist()
if 'daily_return' in num_cols:
    num_cols.remove('daily_return')

corr = gold[num_cols].corr()
print('--- Correlatie Matrix ---')
print(corr.round(3))

# Heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt='.2f',
            mask=mask, square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlatie Tussen Goud en Macro Indicatoren', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cel 10: Correlatie Analyse

# Maandelijkse rendementen verdeling
monthly_returns = gold[col_name].resample('ME').last().pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(monthly_returns, bins=20, color='gold', alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Verdeling Maandelijkse Rendementen')
axes[0].set_xlabel('Maandelijks rendement')
axes[0].set_ylabel('Frequentie')

# Box plot per jaar
yearly_returns = []
for year in range(2020, 2027):
    yr_data = monthly_returns[monthly_returns.index.year == year]
    if len(yr_data) > 0:
        yearly_returns.append((str(year), yr_data.values))

axes[1].boxplot([r[1] for r in yearly_returns], tick_labels=[r[0] for r in yearly_returns])
axes[1].set_title('Maandelijkse Rendementen per Jaar')
axes[1].set_ylabel('Rendement')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print('\n--- Jaarlijks Rendement Statistieken ---')
for year, rets in yearly_returns:
    print(f"{year}: gem={np.mean(rets):.2%}, std={np.std(rets):.2%}, min={np.min(rets):.2%}, max={np.max(rets):.2%}")

## 6. Huidige Portefeuille Status

In [ ]:
# Cel 11: Huidige Portefeuille Status

# Portefeuille waarden berekenen
portfolio_stats = calculate_portfolio_value(portfolio, gold)
goal_stats = goal_progress(portfolio_stats['total_portfolio_value_eur'])

# Volgende aankoop
laatste_aankoop = bars['datum_aankoop'].max()
volgende = next_purchase_date(laatste_aankoop, interval_months=2)

print('=' * 50)
print('   HUIDIGE PORTFOLIO STATUS')
print('=' * 50)
print(f"Aantal goudstaven:      {portfolio_stats['total_bars']}")
print(f"Aantal sieraden:        {portfolio_stats['total_jewelry']}")
print(f"Totaal gewicht:         {format_gram(portfolio_stats['total_weight_gram'])}")
print(f"Zuiver goud:            {format_gram(portfolio_stats['total_pure_gold_gram'])}")
print(f"Totaal geinvesteerd:    {format_eur(portfolio_stats['total_invested_eur'])}")
print(f"Huidige spot prijs:     {format_eur(portfolio_stats['current_spot_eur_gram'])}/gram")
print(f"Huidige waarde:         {format_eur(portfolio_stats['total_portfolio_value_eur'])}")
print(f"ROI:                    {portfolio_stats['roi_percent']:.1f}%" if not np.isnan(portfolio_stats['roi_percent']) else "ROI: N/A")
print()
print(f"DOEL: EUR {goal_stats['goal_eur']:,.0f}")
print(f"Voortgang:              {goal_stats['progress_percent']:.1f}% ({format_eur(goal_stats['current_eur'])} van {format_eur(goal_stats['goal_eur'])})")
print(f"Nog nodig:              {format_eur(goal_stats['remaining_eur'])}")
print(f"Doel bereikt:           {'Ja' if goal_stats['reached'] else 'Nee'}")
print()
print(f"Laatste aankoop:        {laatste_aankoop.strftime('%d/%m/%Y')}")
print(f"Volgende aankoop:       {volgende.strftime('%d/%m/%Y')}")
print('=' * 50)

---
## Samenvatting

**Belangrijkste bevindingen:**
1. Goudprijs toont een opgaande trend sinds 2020
2. Aankoopmomenten zijn zichtbaar op de grafiek - vergelijk met spot prijs
3. Portefeuille is een mix van staven (investering) en sieraden (erfstukken)
4. Correlaties tussen goud en macro indicatoren zijn relevant voor voorspelling

**Volgende stap:** Notebook 04 - Price Prediction (Linear Regression + ARIMA)